This notebook extends the ["Agents"](../../LangChain%20Agents%20%26%20Tools/Labs/Agents.ipynb) example.

In [ ]:
!pip install -q langchain langchain-openai

In [ ]:
from google.colab import userdata
from langchain.agents import create_agent
from langchain.messages import HumanMessage
from langchain.tools import tool
from langchain_core.messages import BaseMessage
from langchain_core.runnables.config import RunnableConfig
from langchain_openai import ChatOpenAI
from langgraph.checkpoint.base import BaseCheckpointSaver
from langgraph.checkpoint.memory import InMemorySaver
from pydantic import SecretStr
from typing import List

openai_api_key = SecretStr(userdata.get('OPENAI_API_KEY'))

def print_conversation(conversation: List[BaseMessage]):
    for message in conversation:
        message.pretty_print()

def explore_checkpoints(checkpoint_saver: BaseCheckpointSaver, runnable_config: RunnableConfig):
    for checkpoint_tuple in checkpoint_saver.list(runnable_config):
        print(checkpoint_tuple)
        print(checkpoint_tuple.checkpoint["channel_values"])

In [ ]:
@tool
def lookup_tour_stop(artist: str) -> str:
    """
    Look up the next city and venue for an artist from a small curated tour calendar.

    Args:
        artist: The artist or band name to search for.
    """
    stops = {
        "breaking benjamin": "Sofia - Arena 8888",
        "placido domingo": "Varna - Palace of Culture and Sports",
        "vassil petrov & jp3": "Shumen - City Stage",
    }
    return stops.get(artist.strip().lower(), "Could not find any tour stops.")

@tool
def estimate_drive_time(origin: str, destination: str) -> str:
    """
    Estimate drive time between cities in Bulgaria.

    Args:
        origin: The departure city.
        destination: The arrival city.
    """
    routes = {
        ("plovdiv", "sofia"): "About 1 hour and 45 minutes.",
        ("shumen", "varna"): "About 1 hour.",
        ("plovdiv", "shumen"): "About 2 hours and 30 minutes."
    }
    key = (origin.strip().lower(), destination.strip().lower())
    return routes.get(key, "Could not estimate the drive time.")

In [ ]:
checkpointer = InMemorySaver()

In [ ]:
agent = create_agent(
    model=ChatOpenAI(model="gpt-5-nano", api_key=openai_api_key),
    tools=[lookup_tour_stop, estimate_drive_time],
    system_prompt="You are a practical live-music concierge. Use tools when they help you give a precise answer.",
    checkpointer=checkpointer
)

In [ ]:
config: RunnableConfig = {
    "configurable": {
        "thread_id": "1"
    }
}

In [ ]:
plan_concert_trip = agent.invoke(
    input={
      "messages": [HumanMessage("I'm in Plovdiv on Friday and want to hear live jazz without wasting the whole evening on travel. Check the current mini tour for \"Vassil Petrov & JP3\", figure out the relevant venue, and estimate the drive time.")]
    },
    config=config
)

In [ ]:
print_conversation(plan_concert_trip["messages"])

In [ ]:
explore_checkpoints(checkpointer, config)

In [ ]:
test_agent_memory = agent.invoke(
    input={
        "messages": [HumanMessage("What do you remember about me?")]
    },
    config=config
)

In [ ]:
print_conversation(test_agent_memory["messages"])

In [ ]:
explore_checkpoints(checkpointer, config)